<a href="https://colab.research.google.com/github/RatchanonPa/Data-Warehouse-and-Big-Data-Analytics/blob/main/LLMs_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install kaggle

In [2]:
import os
import shutil

# อัปโหลด kaggle.json
from google.colab import files
files.upload()  # จะให้คุณเลือกไฟล์ kaggle.json ที่โหลดมา

# สร้างโฟลเดอร์ ~/.kaggle ถ้ายังไม่มี
os.makedirs("/root/.kaggle", exist_ok=True)

# ย้ายไฟล์ไปยังโฟลเดอร์ที่ต้องการ
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")

# ตั้งสิทธิ์การเข้าถึง
os.chmod("/root/.kaggle/kaggle.json", 600)


Saving kaggle.json to kaggle.json


In [3]:
!kaggle competitions download -c crime-charges-analysis

In [4]:
!unzip /content/crime-charges-analysis.zip

Archive:  /content/crime-charges-analysis.zip
  inflating: classes.xlsx            
  inflating: submission.csv          
  inflating: train.csv               


In [5]:
# Discussed conceptually (from PDF notes) and usually run in Colab
!pip install transformers
!pip install datasets
!pip install evaluate -U # -U for upgrade
!pip install accelerate -U
!pip install sentencepiece # For tokenizers like SentencePiece
!pip install gdown # For downloading files from Google Drive

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [6]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Mon May 12 16:17:15 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import pandas as pd

train_df = pd.read_csv('/content/train.csv')
class_df = pd.read_excel('/content/classes.xlsx') # Corrected function name
test_df = pd.read_csv('/content/submission.csv')

In [7]:
train_df

,id,story,answers
0,OhbVrpoiVg,จุงลิ่วขโมยรถจักรยานยนต์จากลานจอด ขณะที่แคสใช้...,"[{'จุงลิ่ว': '[2]'}, {'แคส': '[2,5]'}]"
1,RV5IfLBcbf,จิรศักก์กำลังทำอาหารอยู่ในครัว ในขณะที่ไขนภาเด...,"[{'จิรศักก์': '[1]'}, {'ไขนภา': '[1]'}]"
2,noGMbJmTPS,คมกาญจน์ขโมยเงินสดจากตู้อัตโนมัติในห้างและบุกร...,"[{'คมกาญจน์': '[2,4,5]'}, {'จิรานุวัตร': '[6]'}]"
3,IAoCLrZ3aW,ขจรพงค์ปล่อยให้ระบบลิฟต์โรงแรมขัดข้องจนตกลงมาพ...,[{'ขจรพงค์': '[3]'}]
4,ZkSBvrjn9W,กิมใจยิงเจ้าหน้าที่ขนส่งเสียชีวิตขณะนำของเข้าร...,"[{'กิมใจ': '[2,5,7]'}, {'ขนิฐชีพ': '[5]'}]"
...,...,...,...
1795,uHkBogTZiG,กิติยวดีใช้อาวุธไฟฟ้าช็อต รปภ. ก่อนบุกรุกห้างแ...,"[{'กิติยวดี': '[2,4,5]'}, {'เจติญา': '[2]'}]"
1796,hOKYz1WO6q,กำไลกับจิรวงษ์โจรกรรมในตลาดกลางคืน,"[{'กำไล': '[2]'}, {'จิรวงษ์': '[2]'}]"
1797,1Jk8unffbB,แจ้งฮ้งกำลังนั่งทำการบ้านอยู่ที่บ้านขณะที่ง่วน...,"[{'แจ้งฮ้ง': '[1]'}, {'ง่วนเส็ง': '[1]'}]"
1798,aOXLjbNTI6,จิตต์เกื้อซ่อนมีดในเบาะรถของเกษยุราแล้วจงใจเบร...,"[{'เกษยุรา': '[1]'}, {'จิตต์เกื้อ': '[5,7]'}]"


In [31]:
class_df

,Class,Description
0,1,ไม่มีความผิดหรือไม่มีความเกี่ยวข้องกับเหตุการณ์
1,2,ลักทรัพย์หรือชิงทรัพย์
2,3,กระทำการใดๆด้วยความประมาทหรือละเลยส่งผลให้เกิด...
3,4,บุกรุกเข้าเขตหวงห้ามหรือนอกเวลาทำการ
4,5,กระทำการใดๆด้วยเจตนาที่ส่งผลให้เกิดความเสียหาย...
5,6,"ฉ้อโกงประชาชน, ปลอมแปลงเอกสาร หรือปลอมแปลงสิ่ง..."
6,7,ทำให้มีคนตายด้วยวิธีการใดๆก็ตาม


In [32]:
test_df

,id,story,answers
0,XwlpP5otmI,เคี้ยนแพ้หมากรุกเลยชักปืนขึ้นมายิงจราจนเสียชีวิต,"[{'เคี้ยน': '[5,7]'}, {'จรา': '[1]'}]"
1,1pUXZYyXqQ,จิดาพัฒน์ยิงปืนเล่นน้ำสงกรานต์,[{'จิดาพัฒน์': '[1]'}]
2,drf7jObOL3,กรัณยพรขับรถเมาแล้วชนรถบัสในสถานีขนส่ง ส่งผลให...,"[{'กรัณยพร': '[3,7]'}]"
3,ghD7Mhgjsq,คำนึงภัชทุจริตในการจัดซื้อวัสดุโรงเรียนและปลอม...,NaN
4,5lfeF2x61n,กอบเกิดขับรถขณะหลับในช่วงกลางวัน ทำให้รถพุ่งข้...,NaN
...,...,...,...
995,wEWpIvFoB9,กิมฮัวฉ้อโกงเงินจากลูกค้าผ่านเว็บไซต์ปลอมในงาน...,NaN
996,JbSegK5ioA,กุศลินีบุกรุกเข้าโรงพยาบาล ขโมยยาและทำร้ายพยาบ...,NaN
997,3cnDAIVegR,กลินธ์ขับรถประมาทในการแข่งจนเกิดอุบัติเหตุร้ายแรง,NaN
998,JCTioVnnxg,กชญาณสร้างบริษัทอสังหาริมทรัพย์ปลอม ส่วนจิราทิ...,NaN


In [8]:
import pandas as pd
import ast # For safely evaluating string-represented lists/dicts
from sklearn.preprocessing import MultiLabelBinarizer # For multi-hot encoding

def parse_answers(answer_str):
    if pd.isna(answer_str):
        return []
    try:
        # Safely evaluate the string as a Python literal (list of dicts)
        answers_list = ast.literal_eval(answer_str)
        all_class_ids = set()
        for ans_dict_list in answers_list: # [{'จุงลิ่ว': '[2]'}, {'แคส': '[2,5]'}]
            for entity_ans in ans_dict_list.values(): # e.g., '[2,5]'
                class_ids_str = ast.literal_eval(entity_ans) # e.g., [2,5]
                for class_id in class_ids_str:
                    all_class_ids.add(int(class_id)) # Make sure they are integers
        return sorted(list(all_class_ids))
    except (ValueError, SyntaxError):
        return [] # Handle cases where parsing fails

train_df['parsed_labels'] = train_df['answers'].apply(parse_answers)
train_df

,id,story,answers,parsed_labels
0,OhbVrpoiVg,จุงลิ่วขโมยรถจักรยานยนต์จากลานจอด ขณะที่แคสใช้...,"[{'จุงลิ่ว': '[2]'}, {'แคส': '[2,5]'}]","[2, 5]"
1,RV5IfLBcbf,จิรศักก์กำลังทำอาหารอยู่ในครัว ในขณะที่ไขนภาเด...,"[{'จิรศักก์': '[1]'}, {'ไขนภา': '[1]'}]",[1]
2,noGMbJmTPS,คมกาญจน์ขโมยเงินสดจากตู้อัตโนมัติในห้างและบุกร...,"[{'คมกาญจน์': '[2,4,5]'}, {'จิรานุวัตร': '[6]'}]","[2, 4, 5, 6]"
3,IAoCLrZ3aW,ขจรพงค์ปล่อยให้ระบบลิฟต์โรงแรมขัดข้องจนตกลงมาพ...,[{'ขจรพงค์': '[3]'}],[3]
4,ZkSBvrjn9W,กิมใจยิงเจ้าหน้าที่ขนส่งเสียชีวิตขณะนำของเข้าร...,"[{'กิมใจ': '[2,5,7]'}, {'ขนิฐชีพ': '[5]'}]","[2, 5, 7]"
...,...,...,...,...
1795,uHkBogTZiG,กิติยวดีใช้อาวุธไฟฟ้าช็อต รปภ. ก่อนบุกรุกห้างแ...,"[{'กิติยวดี': '[2,4,5]'}, {'เจติญา': '[2]'}]","[2, 4, 5]"
1796,hOKYz1WO6q,กำไลกับจิรวงษ์โจรกรรมในตลาดกลางคืน,"[{'กำไล': '[2]'}, {'จิรวงษ์': '[2]'}]",[2]
1797,1Jk8unffbB,แจ้งฮ้งกำลังนั่งทำการบ้านอยู่ที่บ้านขณะที่ง่วน...,"[{'แจ้งฮ้ง': '[1]'}, {'ง่วนเส็ง': '[1]'}]",[1]
1798,aOXLjbNTI6,จิตต์เกื้อซ่อนมีดในเบาะรถของเกษยุราแล้วจงใจเบร...,"[{'เกษยุรา': '[1]'}, {'จิตต์เกื้อ': '[5,7]'}]","[1, 5, 7]"


In [9]:
# Use MultiLabelBinarizer to create multi-hot encoded vectors
# Define all possible class IDs (0 to 7 based on class_df)
all_possible_classes = list(range(8)) # Assuming classes are 0-7

mlb = MultiLabelBinarizer(classes=all_possible_classes)
train_labels_multi_hot = mlb.fit_transform(train_df['parsed_labels'])

In [10]:
train_labels_multi_hot

array([[0, 0, 1, ..., 1, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 1, 1, 0],
       ...,
       [0, 1, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 1, 0, 1],
       [0, 0, 1, ..., 0, 1, 0]])

In [11]:
# Add these multi-hot labels back to the DataFrame (optional, but can be useful)
# For direct use with Hugging Face, we might pass this array directly or format it
# For simplicity in this example, let's just keep `parsed_labels` for now
# and assume the model will handle multi-label output.

# For test_df, if you have some answers, process them similarly
# If test_df 'answers' are mostly NaN, you'll predict and then evaluate externally
if 'answers' in test_df.columns:
    test_df['parsed_labels'] = test_df['answers'].apply(parse_answers)
    # test_labels_multi_hot = mlb.transform(test_df['parsed_labels']) # Don't fit again!

In [12]:
# The 'story' column becomes the text input
train_texts = train_df['story'].tolist()
# Convert train_labels_multi_hot to a list of lists of floats for Hugging Face
train_labels_for_hf = [list(map(float, row)) for row in train_labels_multi_hot]

In [14]:
train_texts

['จุงลิ่วขโมยรถจักรยานยนต์จากลานจอด ขณะที่แคสใช้ปืนจี้ชิงเงินสดจากสถานีขนส่ง',
 'จิรศักก์กำลังทำอาหารอยู่ในครัว ในขณะที่ไขนภาเดินตกบันไดด้วยความประมาทส่งผลให้ตนเองขาหัก',
 'คมกาญจน์ขโมยเงินสดจากตู้อัตโนมัติในห้างและบุกรุกเข้าหลังร้านเพื่อขโมยของ มีผลให้เกิดความเสียหายต่อทรัพย์สิน ขณะที่จิรานุวัตรฉ้อโกงผ่านโทรศัพท์',
 'ขจรพงค์ปล่อยให้ระบบลิฟต์โรงแรมขัดข้องจนตกลงมาพร้อมผู้โดยสาร',
 'กิมใจยิงเจ้าหน้าที่ขนส่งเสียชีวิตขณะนำของเข้าร้านทอง และขโมยทองที่บรรทุกมา ส่วนขนิฐชีพทุบทำลายกล้องวงจรปิดด้วยความสะใจเฉยๆ',
 'ขวัญอรุณวางเพลิงอาคารเก่าทำให้ทรัพย์สินเสียหาย และลักทรัพย์ในห้องควบคุม',
 'จิรเวทพิมพ์ผิดรหัสฉุกเฉิน ทำให้ระบบแจ้งเตือนหยุดทำงาน',
 'กาญจน์อรยิงผู้รักษาความปลอดภัยจนเสียชีวิต ขณะที่ขวัญอมรลักบัตรผ่านเข้าออกอาคารและขโมยเอกสารภายใน',
 'กานดาศรีลอบบุกรุกเข้าห้องเก็บของในอาคารและขโมยโทรศัพท์ออกไปในคืนวันหยุด ขณะที่กิมหุนฉ้อโกงนักลงทุนด้วยเอกสารปลอม',
 'กฤติภรใช้มีดข่มขู่ในงานเลี้ยงโรงแรมและทำให้ฝ่ายที่เกี่ยวข้องบาดเจ็บอย่างรุนแรง ขณะที่จำเป็นศรีขโมยของจากตลาดนัดด้วยการบุกรุก',
 'เกรียงกิ

In [13]:
train_labels_for_hf

[[0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0],
 [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0],
 [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 1.0],
 [0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
 [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 1.0, 0.0],
 [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 0.0],
 [0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0],
 [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0],
 [0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0],
 [0.0, 0.0,

In [14]:
# --- 3. Import Hugging Face Libraries ---
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    Trainer,
    TrainingArguments,
)

In [15]:
from datasets import Dataset, DatasetDict
import evaluate
import numpy as np
import torch # For a multi-label problem, BCEWithLogitsLoss is common

In [16]:
# --- 4. Load Pretrained Model, Tokenizer, Config ---
model_name = 'clicknext/phayathaibert' # Or another suitable Thai model
num_classes = 8 # Based on your class_df (0-7)

# For multi-label classification, you often need to set problem_type
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=num_classes,
    problem_type="multi_label_classification" # IMPORTANT for multi-label
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.26M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at clicknext/phayathaibert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
# --- 5. Create DatasetDict ---
# Create a dictionary for the training data
train_data_dict = {'text': train_texts, 'labels': train_labels_for_hf}
hf_train_dataset = Dataset.from_dict(train_data_dict)

In [18]:
# (Create hf_dev_dataset and hf_test_dataset similarly if you have them)
# For now, let's assume you'll split train or use a separate dev file
# Example: if creating a dev set from train_df
# dev_texts = ...
# dev_labels_for_hf = ...
# hf_dev_dataset = Dataset.from_dict({'text': dev_texts, 'labels': dev_labels_for_hf})

dataset_dict_content = {'train': hf_train_dataset}
# if hf_dev_dataset:
#     dataset_dict_content['dev'] = hf_dev_dataset
# if hf_test_dataset: # If you have ground truth for test for evaluation
#     dataset_dict_content['test'] = hf_test_dataset

d = DatasetDict(dataset_dict_content)
d

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 1800
    })
})

In [19]:
print(d)
print(d['train'][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 1800
    })
})
{'text': 'จุงลิ่วขโมยรถจักรยานยนต์จากลานจอด ขณะที่แคสใช้ปืนจี้ชิงเงินสดจากสถานีขนส่ง', 'labels': [0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0]}


In [20]:
# --- 6. Tokenization Function and Mapping ---
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128) # Adjust max_length

tokenized_datasets = d.map(tokenize_function, batched=True)
tokenized_datasets

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1800
    })
})

In [21]:
# Access an element from the 'train' split
tokenized_datasets['train'][0]

{'text': 'จุงลิ่วขโมยรถจักรยานยนต์จากลานจอด ขณะที่แคสใช้ปืนจี้ชิงเงินสดจากสถานีขนส่ง',
 'labels': [0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0],
 'input_ids': [5,
  10,
  7364,
  15948,
  2648,
  3026,
  1398,
  32,
  2707,
  2373,
  2117,
  10773,
  15495,
  2652,
  1895,
  3901,
  32,
  17646,
  6,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,

In [22]:
# --- 7. Define TrainingArguments ---
training_args = TrainingArguments(
    output_dir='./results_legal',
    num_train_epochs=3, # Adjust as needed
    per_device_train_batch_size=16, # Adjust based on GPU memory
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_legal',
    logging_steps=100,
    eval_strategy="epoch", # Changed from evaluation_strategy to eval_strategy
    # eval_steps=500, # if eval_strategy="steps"
    save_strategy="epoch",
    load_best_model_at_end=True, # If using eval_strategy
    metric_for_best_model="f1", # For multi-label, f1-micro or f1-macro is common
    report_to="wandb" # or "none"
)

In [23]:
# --- 8. Define compute_metrics Function (for multi-label) ---
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import EvalPrediction
import torch

def multi_label_metrics(predictions, labels, threshold=0.5):
    # first, use sigmoid on predictions which are logits
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    # next, use threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    # finally, compute metrics
    y_true = labels
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro')
    roc_auc = roc_auc_score(y_true, y_pred, average = 'micro') # Requires probabilities for proper AUC
    accuracy = accuracy_score(y_true, y_pred)
    # return as dictionary
    metrics = {'f1': f1_micro_average,
               'roc_auc': roc_auc,
               'accuracy': accuracy}
    return metrics

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    result = multi_label_metrics(
        predictions=preds,
        labels=p.label_ids)
    return result

In [24]:
# --- 5. Create DatasetDict ---
# Create a dictionary for the training data
train_data_dict = {'text': train_texts, 'labels': train_labels_for_hf}
hf_full_dataset = Dataset.from_dict(train_data_dict)

# Split the dataset into training and evaluation sets
# Adjust the test_size (e.g., 0.1 or 0.2 for 10% or 20% evaluation set)
split_datasets = hf_full_dataset.train_test_split(test_size=0.2)

dataset_dict_content = {
    'train': split_datasets['train'],
    'eval': split_datasets['test'] # Use 'eval' as the key for the evaluation set
}

d = DatasetDict(dataset_dict_content)
print(d) # Verify that 'train' and 'eval' keys exist
print(d['train'][0]) # Check a sample from the training set
print(d['eval'][0])  # Check a sample from the evaluation set

# --- 6. Tokenization Function and Mapping ---
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128) # Adjust max_length

# Apply tokenization to the split datasets
tokenized_datasets = d.map(tokenize_function, batched=True)
tokenized_datasets

# --- 7. Define TrainingArguments ---
training_args = TrainingArguments(
    output_dir='./results_legal',
    num_train_epochs=3, # Adjust as needed
    per_device_train_batch_size=16, # Adjust based on GPU memory
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_legal',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb" # or "none"
)

# --- 8. Define compute_metrics Function (for multi-label) ---
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from transformers import EvalPrediction
import torch

def multi_label_metrics(predictions, labels, threshold=0.5):
    # first, use sigmoid on predictions which are logits
    sigmoid = torch.nn.Sigmoid()
    probs = sigmoid(torch.Tensor(predictions))
    # next, use threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    # finally, compute metrics
    y_true = labels
    # Use zero_division=0 to handle cases where no true or predicted labels exist for a class
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro', zero_division=0)
    # roc_auc_score requires probabilities, not binary predictions
    # Ensure y_true has at least two classes for ROC AUC calculation
    if np.unique(y_true).shape[0] > 1:
        roc_auc = roc_auc_score(y_true, probs, average = 'micro')
    else:
        # Handle cases with only one class in the batch/dataset
        roc_auc = 0.0
        print("Warning: Only one class present in true labels for ROC AUC calculation.")

    accuracy = accuracy_score(y_true, y_pred)
    # return as dictionary
    metrics = {'f1': f1_micro_average,
               'roc_auc': roc_auc,
               'accuracy': accuracy}
    return metrics

def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    result = multi_label_metrics(
        predictions=preds,
        labels=p.label_ids)
    return result

# --- 9. Initialize Trainer ---
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    # Pass the evaluation dataset using the 'eval' key
    eval_dataset=tokenized_datasets["eval"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# --- 10. Start Training ---
# Now the 'eval' dataset exists, so training with evaluation will proceed
trainer.train()

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 1440
    })
    eval: Dataset({
        features: ['text', 'labels'],
        num_rows: 360
    })
})
{'text': 'เจตต์ชัญญาวางยาพิษในอาหารของสามีจนเสียชีวิต เพื่อหวังเงินประกันชีวิต', 'labels': [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0]}
{'text': 'คชรักไม่ตรวจสอบความปลอดภัยก่อนให้ขึ้นลิฟต์ ทำให้ลิฟต์ตก มีผู้เสียชีวิต', 'labels': [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0]}


Map:   0%|          | 0/1440 [00:00<?, ? examples/s]

Map:   0%|          | 0/360 [00:00<?, ? examples/s]

<ipython-input-24-103ac2f48eb2>:88: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ratchanon-faust (ratchanon-faust-khon-kaen-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,F1,Roc Auc,Accuracy
1,No log,0.430748,0.546438,0.908030,0.133333
2,0.572200,0.212854,0.893847,0.979266,0.672222
3,0.290600,0.136520,0.937128,0.991099,0.802778


TrainOutput(global_step=270, training_loss=0.36426290582727505, metrics={'train_runtime': 277.9087, 'train_samples_per_second': 15.545, 'train_steps_per_second': 0.972, 'total_flos': 284175247933440.0, 'train_loss': 0.36426290582727505, 'epoch': 3.0})

In [25]:
# prompt: predict test

import numpy as np
# Assuming 'trainer' object is already trained and available

# --- 11. Prepare Test Data for Prediction ---
# Get the text stories from the test DataFrame
test_texts = test_df['story'].tolist()

# Create a Hugging Face Dataset for the test data
# The test dataset doesn't have ground truth labels for prediction purposes
test_data_dict = {'text': test_texts}
hf_test_dataset = Dataset.from_dict(test_data_dict)

# Tokenize the test dataset
tokenized_test_datasets = hf_test_dataset.map(tokenize_function, batched=True)

# --- 12. Make Predictions on the Test Set ---
# Use the trained trainer to predict on the tokenized test dataset
predictions = trainer.predict(tokenized_test_datasets)

# 'predictions' object contains predictions, label_ids (None for test), and metrics
# Access the logits (raw output before activation)
logits = predictions.predictions

# Apply sigmoid to the logits to get probabilities for multi-label classification
sigmoid = torch.nn.Sigmoid()
probabilities = sigmoid(torch.Tensor(logits)).numpy() # Convert to NumPy array

# Apply a threshold to get binary predictions (0 or 1)
# A common threshold is 0.5, but this can be tuned
threshold = 0.5
binary_predictions = np.zeros(probabilities.shape)
binary_predictions[np.where(probabilities >= threshold)] = 1

# --- 13. Format Predictions for Submission (Optional, depends on submission format) ---
# If you need to output the predicted class IDs for each story, you can do so
# The binary_predictions array has a row for each story and a column for each class (0-7)

predicted_classes = []
for row in binary_predictions:
    # Get the indices where the prediction is 1
    predicted_class_indices = np.where(row == 1)[0].tolist()
    predicted_classes.append(predicted_class_indices)

# Add the predicted classes back to the test_df if needed
test_df['predicted_labels'] = predicted_classes

# You can now inspect test_df or save it
print(test_df[['story', 'predicted_labels']].head())

# You might need to format the predicted_labels column according to the competition's
# submission file requirements (e.g., specific string format).

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

                                               story predicted_labels
0   เคี้ยนแพ้หมากรุกเลยชักปืนขึ้นมายิงจราจนเสียชีวิต        [1, 5, 7]
1                     จิดาพัฒน์ยิงปืนเล่นน้ำสงกรานต์              [1]
2  กรัณยพรขับรถเมาแล้วชนรถบัสในสถานีขนส่ง ส่งผลให...           [3, 7]
3  คำนึงภัชทุจริตในการจัดซื้อวัสดุโรงเรียนและปลอม...           [2, 6]
4  กอบเกิดขับรถขณะหลับในช่วงกลางวัน ทำให้รถพุ่งข้...           [3, 7]


In [26]:
test_df

,id,story,answers,parsed_labels,predicted_labels
0,XwlpP5otmI,เคี้ยนแพ้หมากรุกเลยชักปืนขึ้นมายิงจราจนเสียชีวิต,"[{'เคี้ยน': '[5,7]'}, {'จรา': '[1]'}]","[1, 5, 7]","[1, 5, 7]"
1,1pUXZYyXqQ,จิดาพัฒน์ยิงปืนเล่นน้ำสงกรานต์,[{'จิดาพัฒน์': '[1]'}],[1],[1]
2,drf7jObOL3,กรัณยพรขับรถเมาแล้วชนรถบัสในสถานีขนส่ง ส่งผลให...,"[{'กรัณยพร': '[3,7]'}]","[3, 7]","[3, 7]"
3,ghD7Mhgjsq,คำนึงภัชทุจริตในการจัดซื้อวัสดุโรงเรียนและปลอม...,NaN,[],"[2, 6]"
4,5lfeF2x61n,กอบเกิดขับรถขณะหลับในช่วงกลางวัน ทำให้รถพุ่งข้...,NaN,[],"[3, 7]"
...,...,...,...,...,...
995,wEWpIvFoB9,กิมฮัวฉ้อโกงเงินจากลูกค้าผ่านเว็บไซต์ปลอมในงาน...,NaN,[],[6]
996,JbSegK5ioA,กุศลินีบุกรุกเข้าโรงพยาบาล ขโมยยาและทำร้ายพยาบ...,NaN,[],"[2, 4, 5]"
997,3cnDAIVegR,กลินธ์ขับรถประมาทในการแข่งจนเกิดอุบัติเหตุร้ายแรง,NaN,[],[3]
998,JCTioVnnxg,กชญาณสร้างบริษัทอสังหาริมทรัพย์ปลอม ส่วนจิราทิ...,NaN,[],[6]
